<a href="https://colab.research.google.com/github/tomonari-masada/course2026-nlp/blob/main/04_text_classification_with_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 言語モデルを使ったテキスト分類
* テキスト分類はBoWでも良い性能を出せることが多い。
  * 言語モデルを使って文書分類するときは・・・
  * BoW+SVMをベースラインとして設定して・・・
  * 言語モデルを使った場合と性能比較した方が良い。
* 今回はトランスフォーマ言語モデルを使ってみる。
  * パラメータ数は数億個オーダのもの。
  * パラメータ数が数十億個（数ビリオン）のものは、扱いがやや大変。

* ランタイムのタイプをGPUに設定しておく。

## 準備

In [ ]:
from transformers import set_seed

set_seed(0)

## データセット
* ライブドアニュースコーパスを使う。

In [ ]:
from datasets import load_dataset

ds = load_dataset("mteb/LivedoorNewsClustering")

In [ ]:
ds["train"][0]

In [ ]:
category_names = [
    'dokujo-tsushin',
    'it-life-hack',
    'kaden-channel',
    'livedoor-homme',
    'movie-enter',
    'peachy',
    'smax',
    'sports-watch',
    'topic-news',
]

## テキストの埋め込み

###  埋め込みのための言語モデル
* 下のリーダボードから性能が良さそうなものを選ぶ。
  * https://huggingface.co/spaces/mteb/leaderboard
* ただし、日本語対応しているものを選ぶ。

* `cl-nagoya/ruri-v3-310m`を使ってみる。
  * https://huggingface.co/cl-nagoya/ruri-v3-310m

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("cl-nagoya/ruri-v3-310m")

In [ ]:
model

* 言語モデルの本体
  * テキストを埋め込んでいる間に解説する。

In [ ]:
list(model.modules())[2]

* 例えば、token埋め込みの重みを見てみる。

In [ ]:
list(model.modules())[2].embeddings.tok_embeddings.weight

### 埋め込みの実行
* 訓練セットの埋め込みは・・・
  * Google Colab無料版で10分弱。
  * iMac Apple M3で18分。
  * RTX4090で50秒(!)

In [ ]:
# 計算時間を短縮するため、トークン列の最大長を制限する。
model.max_seq_length = 128

In [ ]:
train_embeddings = model.encode(ds["train"]["sentences"], show_progress_bar=True)

* Google Driveの適当なディレクトリに保存する。

In [ ]:
import os
import numpy as np

data_dir = "/content/drive/MyDrive/2026courses/nlp/livedoor/gpu"

with open(os.path.join(data_dir, 'train_embeddings.npy'), 'wb') as f:
  np.save(f, train_embeddings)

* 検証セットとテストセットも埋め込む。

In [ ]:
validation_embeddings = model.encode(ds["validation"]["sentences"], show_progress_bar=True)
test_embeddings = model.encode(ds["test"]["sentences"], show_progress_bar=True)

In [ ]:
with open(os.path.join(data_dir, 'val_embeddings.npy'), 'wb') as f:
  np.save(f, validation_embeddings)
with open(os.path.join(data_dir, 'test_embeddings.npy'), 'wb') as f:
  np.save(f, test_embeddings)

### 埋め込みベクトルの読み込み

In [ ]:
import os
import numpy as np

data_dir = "/content/drive/MyDrive/2026courses/nlp/livedoor/gpu"

with open(os.path.join(data_dir, 'train_embeddings.npy'), 'rb') as f:
  train_embeddings = np.load(f)
with open(os.path.join(data_dir, 'val_embeddings.npy'), 'rb') as f:
  validation_embeddings = np.load(f)
with open(os.path.join(data_dir, 'test_embeddings.npy'), 'rb') as f:
  test_embeddings = np.load(f)

In [ ]:
train_embeddings.shape

## SVMによる分類

* 訓練セットと検証をセットを結合する。
  * 後で交差検証をするため。

In [ ]:
embeddings = np.concatenate([train_embeddings, validation_embeddings])
labels = np.array(list(ds["train"]["labels"]) + list(ds["validation"]["labels"]))

* 交差検証によりハイパーパラメータをチューニングする。

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

for C in 10. ** np.arange(-1, 4):
  print(f"C={C:.2e}")
  scores = []
  skf_split = skf.split(embeddings, labels)
  print("\tsplit-wise accuracies:", end="")
  for train_index, val_index in skf_split:
    X_train = embeddings[train_index]
    clf = LinearSVC(C=C, dual=False, max_iter=1000, random_state=123)
    clf.fit(X_train, labels[train_index])
    X_val = embeddings[val_index]
    score = clf.score(X_val, labels[val_index])
    print(f"\t{score:.3f}", end="")
    scores.append(score)
  print(f"\n\tmean accuracy: {np.array(scores).mean():.3f}")

* 選ばれた設定をテストセットで評価する。

In [ ]:
from sklearn.svm import LinearSVC

clf = LinearSVC(C=10.0, dual=False, max_iter=1000, random_state=123)
clf.fit(embeddings, labels)
score = clf.score(test_embeddings, ds["test"]["category"])
print(f"{score:.3f}")

# 課題
* 作成済みの埋め込みベクトルを使って・・・
* 分類手法（SVM以外でも構わない）だけをチューニングすることで・・・
* どのくらい分類性能を上げられるか、試行錯誤してみてください。
  * 今回の課題では、そんなに良い分類性能は出せないと思います。
  * パラメータ数が数億個の言語モデルであれば、ファインチューニングして使うのが、通常です。
  * その方法は、いずれ説明します。